# Lab 1.2 – Hybrid Retrieval + Custom Re-ranker Pipeline

**Course:** Advanced RAG Architecture, Custom Skills & Evaluation on Google Cloud Platform  
**Module:** Week 1 – Precision RAG & Domain Chunking for Compliance Data

This notebook builds on the enriched corpus from Lab 1.1 and implements:

1. BM25 sparse retrieval
2. Dense retrieval (HashingEmbedder offline / Vertex AI when available)
3. Reciprocal Rank Fusion (RRF)
4. Lightweight domain-aware re-ranker
5. Precision@3 and hit-rate@3 evaluation against a held-out DFD query set

The primary success metric is **hit-rate@3** (did we surface the correct control in the top-3?). Precision@3 is reported for completeness.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

LAB_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(LAB_ROOT))

from src.chunking import process_directory
from src.ingest import write_jsonl
from src.retrieval import (
    HybridRetriever,
    HashingEmbedder,
    load_eval_queries,
    evaluate_retriever,
)

DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Lab root : {LAB_ROOT}")
print(f"Data dir : {DATA_DIR}")

## 1. Build / load the Lab 1.1 corpus

In [ ]:
corpus = process_directory(DATA_DIR)
jsonl_path = OUTPUT_DIR / "rag_chunks.jsonl"
write_jsonl(corpus, jsonl_path)
print(f"Corpus size: {len(corpus)} chunks → {jsonl_path}")

## 2. Instantiate the hybrid retriever

By default we use `HashingEmbedder` so the notebook runs without GCP credentials.  
Replace with `VertexEmbedder(project_id=...)` when you want real embeddings.

In [ ]:
retriever = HybridRetriever.from_jsonl(
    jsonl_path,
    embedder=HashingEmbedder(dim=256),
)
print(f"Ready – {len(retriever.records)} documents indexed (BM25 + dense)")

## 3. Single-query walkthrough

In [ ]:
query = "Can an external entity write directly to an internal data store?"
result = retriever.retrieve(query, top_k=5)

print(f"Query: {query}\n")
print("=== BM25 top-3 ===")
for h in result.bm25_hits[:3]:
    print(f"  [{h.rank}] {h.control_id or '—':12}  score={h.score:.3f}  {h.record.section[:50]}")

print("\n=== Dense top-3 ===")
for h in result.dense_hits[:3]:
    print(f"  [{h.rank}] {h.control_id or '—':12}  score={h.score:.3f}  {h.record.section[:50]}")

print("\n=== Final (RRF + re-rank) top-5 ===")
for h in result.final_hits:
    print(f"  [{h.rank}] {h.control_id or '—':12}  score={h.score:.3f}  {h.record.section[:50]}")

## 4. Metadata filtering

In [ ]:
result = retriever.retrieve(
    "trust boundary",
    top_k=5,
    asset_type="trust_boundary",
)
print("Filters applied:", result.filters_applied)
for h in result.final_hits:
    print(f"  {h.control_id or '—':12}  asset={h.record.asset_type:16}  {h.record.section[:45]}")

## 5. Systematic evaluation – precision@3 and hit-rate@3 across modes

We compare four configurations on the same 12 evaluation queries.  
**Hit-rate@3** is the primary metric (did the correct control appear in the top-3?).

In [ ]:
queries = load_eval_queries(DATA_DIR / "evaluation_queries.json")
print(f"Loaded {len(queries)} evaluation queries\n")

configs = {
    "BM25 only":        dict(use_dense=False, use_bm25=True,  use_rerank=False),
    "Dense only":       dict(use_dense=True,  use_bm25=False, use_rerank=False),
    "Hybrid (RRF)":     dict(use_dense=True,  use_bm25=True,  use_rerank=False),
    "Hybrid + re-rank": dict(use_dense=True,  use_bm25=True,  use_rerank=True),
}

print(f"{'Mode':20s}  {'P@3':>6}  {'Hit@3':>6}")
print("-" * 36)
for name, flags in configs.items():
    metrics = evaluate_retriever(retriever, queries, k=3, **flags)
    p = metrics["mean_precision_at_3"]
    h = metrics["mean_hit_rate_at_3"]
    print(f"{name:20s}  {p:6.3f}  {h:6.3f}")

print("\n✓ Hybrid paths should show the strongest hit-rate@3.")

## 6. Per-query detail for the hybrid configuration

In [ ]:
best = evaluate_retriever(
    retriever, queries, k=3,
    use_dense=True, use_bm25=True, use_rerank=True,
)

print(f"Mean precision@3 = {best['mean_precision_at_3']:.3f}")
print(f"Mean hit-rate@3  = {best['mean_hit_rate_at_3']:.3f}\n")
for row in best["per_query"]:
    status = "✓" if row["hit_rate_at_k"] >= 1.0 else "·"
    print(
        f"{status} {row['query_id']}  hit={row['hit_rate_at_k']:.0f}  "
        f"P@3={row['precision_at_k']:.2f}  "
        f"got={row['retrieved_control_ids']}  "
        f"expected={row['expected_control_ids']}"
    )

## 7. Design notes (for your submission)

Record observations here:

- When does BM25 dominate? (exact control IDs, rare terms)
- When does dense help? (paraphrases, conceptual questions)
- Effect of the re-ranker boosts
- Any changes you made to `k`, boost values, or fusion strategy

The reference implementation uses a pure-Python BM25, a hashing embedder (offline), classic RRF with k=60, and a simple deterministic re-ranker. All of these are intentionally auditable for security workloads.

## Success criteria checklist

- [x] Hybrid pipeline runs end-to-end offline
- [x] Hit-rate@3 and precision@3 measured for four retrieval modes
- [x] Metadata filters demonstrated
- [x] Evaluation queries + ground truth provided
- [x] Code reusable via `HybridRetriever`

**Next:** Week 2 – Storage Architecture on GCP (Vertex AI Vector Search, AlloyDB pgvector, BigQuery, Graph).